# Übung 02: Python-Grundlagen

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 2 (Python-Grundlagen)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Datentypen & Datenstrukturen, Kontrollstrukturen, Funktionen, Fehlerbehandlung, String-Methoden, f-Strings, Regex, Datum & Zeit |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 210 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Der fiktive *Bahnhofkiosk Winterthur* exportiert sein Kassenjournal als Textdatei (eine Zeile pro Buchung: `Datum Zeit;Artikel;Menge;Preis`). Die Formate sind uneinheitlich – wie oft bei echten Exporten.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import re
import json
from datetime import datetime, date, timedelta

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung02'
# ---------------------------------------------------------------
import os
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(2026)
os.makedirs('data_uebung02', exist_ok=True)

articles = {  # name: (price, weight)
    'Kaffee gross': (4.80, 14), 'Kaffee klein': (3.90, 12), 'Gipfeli Butter': (1.20, 14),
    'Sandwich Schinken': (6.90, 8), 'Zeitung NZZ': (4.50, 6), 'Zeitung 20min': (0.00, 5),
    'Mineral 5dl': (2.90, 9), 'Kaugummi': (2.50, 6), 'Autobahnvignette': (40.00, 1),
    'Prepaid-Karte 20': (20.00, 2), 'Geschenkkarte 1000': (1000.00, 0.25),
    'Schoggi Riegel': (1.90, 8), 'Lottoschein': (3.00, 3),
}
names = list(articles)
w = np.array([articles[n][1] for n in names], dtype=float)
w = w / w.sum()

def fmt_price(p, i):
    if p == 0:
        return 'gratis'
    whole = f'{p:,.2f}'.replace(',', "'")
    if p >= 1000 and i % 2 == 0:
        whole = whole.replace("'", '’')                  # typographic apostrophe
    if p == int(p) and p < 1000:
        variant = i % 3
        if variant == 0:
            return f'Fr. {int(p)}.–'
        if variant == 1:
            return f'CHF {int(p)}.-'
    return f'CHF {whole}'

def fmt_qty(q, i):
    return [f'{q}x', f'{q} x', f'{q} Stk', f'{q}'][i % 4]

lines, total, n_lines = [], 0.0, 0
day = date(2026, 10, 1)
while day <= date(2026, 10, 31):
    if day.weekday() != 6:                               # closed on Sundays
        for k in range(int(rng.integers(18, 30))):
            n = names[rng.choice(len(names), p=w)]
            price = articles[n][0]
            q = int(rng.choice([1, 1, 1, 1, 2, 2, 3, 4, 10, 12])) if price < 10 else 1
            hh, mm = int(rng.integers(6, 20)), int(rng.integers(0, 60))
            lines.append(f'{day:%d.%m.%Y} {hh:02d}:{mm:02d};{n};{fmt_qty(q, n_lines)};{fmt_price(price, n_lines)}')
            total += q * price
            n_lines += 1
    day += timedelta(days=1)

with open('data_uebung02/kassenjournal_oktober.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')

notes = [
    'Kunde bestellt 3x ART-10422 und 12 Stk ART-00917, Abholung 14:30',
    'Nachlieferung: 2 x ART-55001; Rückfrage unter 052 123 45 67',
    'ART-10422 (1x) für Herrn Meier, bitte 10 Stk. ART-00917 dazu',
    'Storno! Nicht liefern: ART-77777',
    '5x ART-55001, 1 x ART-10422 – Rechnung an Kiosk Seen',
    'Bestellung Firma Huber: 24 Stk ART-00917 / 6x ART-31415',
]
with open('data_uebung02/bestellnotizen.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(notes) + '\n')

print('Files created:', sorted(os.listdir('data_uebung02')))
print(f'Kassenjournal: {n_lines} Zeilen')
print(f"Kassenabschluss Oktober (Kontrollwert): CHF {total:,.2f}".replace(',', "'"))

### Daten ansehen

Die ersten Zeilen des Kassenjournals. Öffnen Sie die Datei auch im Editor, um alle Formatvarianten zu sehen.

In [ ]:
# Show the first 12 lines of the cash journal
with open('data_uebung02/kassenjournal_oktober.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i == 12:
            break
        print(line.rstrip())

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 48 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Code bzw. zu den Kioskdaten werden nicht gewertet.*

### A1 (4 Punkte)

Notieren Sie **zuerst ohne Ausführen**, was der folgende Code ausgibt. Führen Sie ihn danach aus und erklären Sie die **zwei** Ergebnisse, die Sie am ehesten überrascht haben bzw. die für Datenanalysen am gefährlichsten sind. *(→ 2.1, 2.4)*

```python
print(7 // 2, -7 // 2, 7 % 3)
print(round(2.5), round(3.5), round(0.125, 2))
print(bool([]), bool('0'), bool(0.0))
print('5' * 3, [0] * 3)
x = None
print(x == None, x is None)
```

**Ihre Vorhersage und Erklärung:**

> [Hier eintragen]

In [ ]:
# Code zum Überprüfen hier eintragen

### A2 (4 Punkte)

Ein Kollege schreibt folgenden Code, um eine Preisliste zu «sichern», bevor er eine Aktion einrechnet:

```python
preise_original = [4.80, 3.90, 1.20]
preise_aktion = preise_original
for i in range(len(preise_aktion)):
    preise_aktion[i] = round(preise_aktion[i] * 0.9, 2)
print(preise_original)
```

a) Was wird ausgegeben und warum?

b) Wie korrigieren Sie den Code? Welche Entsprechung gibt es in pandas, wenn man einen Data Frame für eine Aufgabe weiterverwendet (z.B. `df_B1 = df_raw[...]`)? *(→ 6.1)*

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Welche Datenstruktur (`list`, `tuple`, `dict`, `set`) wählen Sie für die folgenden drei Fälle im Kiosk? Begründen Sie jeweils in einem Satz. *(→ 2.1)*

1. Nachschlagen des Preises zu einem Artikelnamen (z.B. `'Kaffee gross'` → `4.80`).
2. Ermitteln, welche Kundenkarten-Nummern im September **und** im Oktober eingekauft haben.
3. Speichern der Öffnungszeit als unveränderliches Paar `(06:00, 20:00)`, das als Schlüssel in einem Dictionary dienen soll.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis. Führen Sie den Code aus, um das tatsächliche Ergebnis zu sehen.*

### B1 (8 Punkte)

**Ziel:** Den Oktober-Umsatz des Kiosks aus dem Kassenjournal berechnen (Summe über alle Zeilen von Menge × Preis).

Der **Kassenabschluss** (Kontrollwert) wird beim Ausführen des Setups ausgegeben. Das Ergebnis des folgenden Codes weicht deutlich davon ab.

In [ ]:
# Original code (do not modify this cell)
def parse_price(text):
    try:
        return float(text.replace('CHF', '').strip())
    except:
        return 0.0

def parse_qty(text):
    return int(text[0])

umsatz = 0
with open('data_uebung02/kassenjournal_oktober.txt', 'r', encoding='utf-8') as f:
    for line in f:
        ts, artikel, menge, preis = line.strip().split(';')
        umsatz += parse_qty(menge) * parse_price(preis)

print(f'Umsatz Oktober: CHF {umsatz:,.2f}')

**Aufgabe:** Finden Sie **alle** Ursachen für die Abweichung (Tipp: Sehen Sie sich die verschiedenen Formate in den Spalten Menge und Preis systematisch an, z.B. mit einem `set()` der vorkommenden Muster). Schreiben Sie den korrigierten Code, sodass der Umsatz mit dem Kassenabschluss übereinstimmt (auf den Rappen genau). Begründen Sie jede Änderung. *(→ 2.2, 2.3, 2.5, 11.2)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Parse functions

# Sum over all lines

# Show the result

### B2 (8 Punkte)

**Ziel:** Umsatz pro Wochentag, um die Personalplanung zu verbessern.

Der Kiosk ist **sonntags geschlossen**, und er hatte im Oktober an **jedem** anderen Tag geöffnet.

In [ ]:
# Original code (do not modify this cell)
weekday_names = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
lines_per_weekday = {name: 0 for name in weekday_names}
n_skipped = 0

with open('data_uebung02/kassenjournal_oktober.txt', 'r', encoding='utf-8') as f:
    for line in f:
        ts = line.split(';')[0]
        try:
            d = datetime.strptime(ts, '%m.%d.%Y %H:%M')
        except ValueError:
            n_skipped += 1
            continue
        lines_per_weekday[weekday_names[d.weekday()]] += 1

print('Anzahl Buchungen pro Wochentag:', lines_per_weekday)
print('Übersprungene Zeilen:', n_skipped)

**Aufgabe:** Erklären Sie, warum das Ergebnis Buchungen am Sonntag zeigt und warum Zeilen übersprungen werden. Schreiben Sie den korrigierten Code und begründen Sie jede Änderung. Prüfen Sie Ihr Ergebnis mit einer Plausibilitätsüberlegung (Anzahl Zeilen gesamt, Sonntag). *(→ 2.6, 1.3)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Count lines per weekday

# Show the result

---
## Teil C – Python-Code schreiben (20 Punkte)

*Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Lesen Sie das Kassenjournal ein und erstellen Sie eine **Liste von Dictionaries** (eine pro Zeile) mit den Schlüsseln `datum` (Typ `date`), `artikel` (Text) und `menge` (Ganzzahl). Geben Sie anschliessend die **5 Artikel mit der grössten verkauften Menge** als formatierte Tabelle aus: Artikelname linksbündig mit Breite 20, Menge rechtsbündig mit Breite 6, z.B.

```
Artikel               Menge
Kaffee gross            ...
```

Begründen Sie, wie Sie die Mengen aggregieren und sortieren. *(→ 2.1, 2.4)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Die Datei `data_uebung02/bestellnotizen.txt` enthält Freitext-Notizen. Extrahieren Sie mit **einem** regulären Ausdruck alle Paare aus **Menge und Artikelnummer** (Format `ART-` + 5 Ziffern), wobei die Menge vor der Artikelnummer steht und als `3x`, `2 x`, `12 Stk` oder `10 Stk.` geschrieben sein kann. Summieren Sie die Mengen pro Artikelnummer in einem Dictionary und geben Sie es sortiert nach Artikelnummer aus.

Begründen Sie, warum Ihr Muster `ART-10422 (1x)` (Menge nach der Nummer) und `ART-77777` (Storno ohne Menge) **nicht** erfasst, und ob das fachlich gewünscht ist. *(→ 2.5)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Schreiben Sie eine Funktion `normalize_phone(text)`, die Schweizer Festnetz- und Mobilnummern in das Format `+41 52 123 45 67` bringt und für ungültige Eingaben `None` zurückgibt. Verwenden Sie **keine** nackte `except:`-Klausel. Die Funktion muss folgende Fälle korrekt behandeln:

| Eingabe | Erwartete Ausgabe |
|---|---|
| `'052 123 45 67'` | `'+41 52 123 45 67'` |
| `'0521234567'` | `'+41 52 123 45 67'` |
| `'+41 79 555 66 77'` | `'+41 79 555 66 77'` |
| `'0041791234567'` | `'+41 79 123 45 67'` |
| `'052 123 45'` | `None` |
| `None` | `None` |

Testen Sie Ihre Funktion mit einer Schleife über alle Fälle und geben Sie pro Fall `OK` oder `FEHLER` aus. Begründen Sie, wie Ihre Funktion ungültige Eingaben erkennt. *(→ 2.2, 2.3, 2.5)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Verwenden Sie als «heute» fix den **19.10.2026** (nicht `date.today()`, damit das Ergebnis reproduzierbar ist).

1. Berechnen Sie, wie viele Tage es bis zum LN 1 am **09.11.2026** dauert.
2. Geben Sie für die nächsten **drei Montage** ab heute (heute nicht mitgezählt) das Datum im Format `TT.MM.JJJJ` aus.
3. Lesen Sie die Liste `liefertermine = ['31.10.2026', '02.11.2026', '30.02.2026', '15.11.26']` ein, wandeln Sie jeden gültigen Eintrag in ein `date` um und geben Sie für ungültige Einträge eine verständliche Meldung aus (statt abzustürzen). Geben Sie für jeden gültigen Termin den Wochentag auf **Deutsch** aus.

Begründen Sie, wie Sie den deutschen Wochentag bestimmen, ohne von den Spracheinstellungen des Computers abhängig zu sein. *(→ 2.6)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Gegeben sind die Kundenkarten-Nummern, mit denen im September bzw. im Oktober eingekauft wurde:

```python
karten_sep = ['K-1001', 'K-1002', 'K-1003', 'K-1003', 'K-1007', 'K-1010', 'K-1011']
karten_okt = ['K-1002', 'K-1003', 'K-1004', 'K-1010', 'K-1010', 'K-1012']
```

Berechnen Sie mit Mengenoperationen: (a) Anzahl **verschiedener** Karten pro Monat, (b) Karten, die in **beiden** Monaten aktiv waren, (c) **verlorene** Karten (nur September) und (d) **neue** Karten (nur Oktober). Berechnen Sie zusätzlich die **Retention-Rate** = Anteil der September-Karten, die auch im Oktober aktiv waren, und geben Sie sie als Prozentzahl mit einer Nachkommastelle aus (z.B. `42.9%`).

Begründen Sie, warum hier `set` und nicht `list` die richtige Struktur ist. *(→ 2.1, 2.4)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')